In [5]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [6]:
from datasets import load_dataset

# Stream one English sample — no full download
ds = load_dataset(
    "amphion/Emilia-Dataset",
    split="train",
    streaming=True,
)
sample = next(iter(ds))

print("Keys:", list(sample.keys()))
print("Text:", sample.get("text", sample.get("json", {}).get("text", "—")))

audio = sample["mp3"]  # dict with 'array' and 'sampling_rate'
print(
    f"Sample rate: {audio['sampling_rate']} Hz, length: {len(audio['array'])} samples"
)

Resolving data files:   0%|          | 0/4343 [00:00<?, ?it/s]

Keys: ['json', 'mp3', '__key__', '__url__']
Text:  So. Chloe hat gesagt, ich soll noch unten gehen. Was ich natürlich auch machen werde.
Sample rate: 24000 Hz, length: 193968 samples


In [7]:
import torch

# transform audio to tensor and add dimension
audio_tensor = torch.tensor(audio["array"]).unsqueeze(0).float()
sr = audio["sampling_rate"]
print(f"Audio tensor: {audio_tensor.shape}, sr={sr}")

Audio tensor: torch.Size([1, 193968]), sr=24000


In [8]:
from TTS.api import TTS

# bypass coqui aggrement
os.environ["COQUI_TOS_AGREED"] = "1"
# Downloads and caches to ~/.local/share/tts/ on first run
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
model = tts.synthesizer.tts_model
print("Model loaded:", type(model).__name__)

C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\jsonlines\jsonlines.py:324: SyntaxWarning: invalid escape sequence '\*'
  :param \*\*kwargs: additional arguments, forwarded to the reader or writer
C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\pysbd\segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\pysbd\lang\arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\pysbd\lang\persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Model loaded: Xtts


In [9]:
gpt_cond_latent = model.get_gpt_cond_latents(
    audio_tensor, sr, length=model.config.gpt_cond_len
)
speaker_embedding = model.get_speaker_embedding(audio_tensor, sr)

print("gpt_cond_latent shape:", gpt_cond_latent.shape)
print("speaker_embedding shape:", speaker_embedding.shape)

gpt_cond_latent shape: torch.Size([1, 32, 1024])
speaker_embedding shape: torch.Size([1, 512, 1])


---

# Temporary Example extracting "directions" for pitch etc.

In [10]:
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import numpy as np
import torch
import librosa


def load_or_generate_directions(
        *,
        create_new: bool,
        ds: Optional[Iterable[dict]] = None,
        model: Optional[Any] = None,
        max_scan: int = 1000,
        pct: float = 0.15,
        save_dir: str | Path = "directions",
        load_path: str | Path = None,
        save: bool = True,
        filename_prefix: str = "xtts_directions",
        device: Optional[str] = None,
        include_speaker: bool = True,  # <--- NEW: also learn directions in speaker-embedding space
) -> Dict[str, Any]:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if not create_new:
        load_path = Path(load_path)
        payload = torch.load(load_path, map_location=device)
        print(f"Loaded directions from: {load_path}")
        return payload

    # --- Collect representations + stats ---
    all_latents: list[torch.Tensor] = []
    all_spk_embs: list[torch.Tensor] = []  # <--- NEW
    stats: list[dict] = []

    print(f"Scanning {max_scan} samples... (pct={pct:.3f}, device={device}, include_speaker={include_speaker})")

    processed = 0
    for i, sample in enumerate(ds):
        if i >= max_scan:
            break

        audio_array = np.asarray(sample["mp3"]["array"])
        sr = int(sample["mp3"]["sampling_rate"])
        text = sample.get("text", "") or sample.get("json", {}).get("text", "") or ""

        duration = len(audio_array) / sr if sr > 0 else 0.0
        cps = (len(text) / duration) if duration > 0 else 0.0

        f0 = librosa.yin(
            audio_array,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
        )
        mean_pitch = float(np.mean(f0[f0 > 0])) if np.any(f0 > 0) else 0.0

        audio_tensor = torch.tensor(audio_array).unsqueeze(0).float().to(device)
        with torch.no_grad():
            latent = model.get_gpt_cond_latents(
                audio_tensor,
                sr,
                length=model.config.gpt_cond_len,
            )
            all_latents.append(latent)

            if include_speaker:
                spk = model.get_speaker_embedding(audio_tensor, sr)
                all_spk_embs.append(spk)

        stats.append({"cps": float(cps), "pitch": float(mean_pitch)})
        processed += 1

        if (processed % 25 == 0):
            print(f"  Processed {processed}/{max_scan}...")

    if processed == 0:
        raise RuntimeError("Processed 0 samples; dataset iterator may be empty or failing")

    def _direction_for(feature_name: str, reps: list[torch.Tensor]) -> torch.Tensor:
        values = np.array([s[feature_name] for s in stats], dtype=np.float64)
        indices = np.argsort(values)

        num = max(1, int(processed * pct))
        low_idx = indices[:num]
        high_idx = indices[-num:]

        print(f"Calculating {feature_name}: averaging {len(low_idx)} low vs {len(high_idx)} high samples")

        mean_low = torch.stack([reps[int(j)] for j in low_idx]).mean(dim=0)
        mean_high = torch.stack([reps[int(j)] for j in high_idx]).mean(dim=0)
        return mean_high - mean_low

    # GPT-latent directions (existing behavior)
    speed_direction = _direction_for("cps", all_latents)
    pitch_direction = _direction_for("pitch", all_latents)

    payload: Dict[str, Any] = {
        "speed_direction": speed_direction.detach(),
        "pitch_direction": pitch_direction.detach(),
        # Helpful to keep around if you want to compute anchor points later without re-scanning:
        "stats": stats,
        "all_latents": [t.detach().cpu() for t in all_latents],
    }

    # Speaker-embedding directions (NEW)
    if include_speaker:
        spk_speed_direction = _direction_for("cps", all_spk_embs)
        spk_pitch_direction = _direction_for("pitch", all_spk_embs)
        payload.update(
            {
                "spk_speed_direction": spk_speed_direction.detach(),
                "spk_pitch_direction": spk_pitch_direction.detach(),
                "all_spk_embs": [t.detach().cpu() for t in all_spk_embs],
            }
        )

    created_at = datetime.now().strftime("%Y%m%d_%H%M%S")
    pct_label = f"{pct * 100:.1f}".replace(".", "p")  # e.g., 15.0 -> "15p0"
    filename = f"{filename_prefix}_N{processed}_pct{pct_label}_{created_at}.pt"

    if save:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / filename
        torch.save(payload, save_path)
        print(f"Saved directions to: {save_path}")

    return payload


In [11]:
CREATE_NEW = False
SAVE = False

directions = load_or_generate_directions(
    create_new=CREATE_NEW,
    save=SAVE,
    ds=ds,
    model=model,
    max_scan=1000,
    pct=0.15,
    save_dir="directions",
    load_path=r"directions\xtts_directions_N1000_pct15p0_20260225_162051.pt",
    include_speaker=True,  # <--- NEW: attempt to load/compute speaker-space directions too
)

speed_direction = directions["speed_direction"]
pitch_direction = directions["pitch_direction"]

# NEW: speaker-space directions (may not exist in older .pt files)
spk_speed_direction = directions.get("spk_speed_direction", None)
spk_pitch_direction = directions.get("spk_pitch_direction", None)

Loaded directions from: directions\xtts_directions_N1000_pct15p0_20260225_162051.pt


In [12]:
from ipywidgets import widgets
from IPython.display import Audio, display, clear_output

# UI Elements (shared)
s_slider = widgets.FloatSlider(value=0, min=-5, max=5, step=0.5, description="Speed 🏃")
p_slider = widgets.FloatSlider(value=0, min=-5, max=5, step=0.5, description="Pitch 🎤")

# NEW: switch between modifying GPT conditioning vs speaker embedding
apply_to = widgets.ToggleButtons(
    options=[("Conditional Latent", "gpt"), ("Speaker Embedding", "spk")],
    value="gpt",
    description="Apply to:",
)

linear_btn = widgets.Button(description="Generate Linear", button_style="info", icon="play")
spherical_btn = widgets.Button(description="Generate Spherical", button_style="warning", icon="play")

ui_out = widgets.Output()


# --- Spherical interpolation helper ---
def slerp(val, low, high):
    """
    Spherical linear interpolation between two tensors (low and high)
    based on 'val' (0.0 to 1.0).
    """
    low_norm = low / torch.norm(low)
    high_norm = high / torch.norm(high)

    dot = torch.sum(low_norm * high_norm)
    dot = torch.clamp(dot, -1.0, 1.0)
    omega = torch.acos(dot)
    so = torch.sin(omega)

    if so < 1e-6:
        return (1.0 - val) * low + val * high

    return (torch.sin((1.0 - val) * omega) / so) * low + (torch.sin(val * omega) / so) * high


# NEW: pull scan artifacts (if present) for anchor-point computation
stats = directions.get("stats", None)
all_latents = directions.get("all_latents", None)
all_spk_embs = directions.get("all_spk_embs", None)


def get_anchor_points(feature_name: str, reps: list[torch.Tensor], stats_list: list[dict], *, pct: float = 0.15):
    values = [s[feature_name] for s in stats_list]
    indices = np.argsort(values)
    num = max(1, int(len(indices) * pct))

    low_idx = indices[:num]
    high_idx = indices[-num:]

    print(f"Anchoring {feature_name}: using {len(low_idx)} samples for Low and High points")

    mean_low = torch.stack([reps[int(i)] for i in low_idx]).mean(dim=0)
    mean_high = torch.stack([reps[int(i)] for i in high_idx]).mean(dim=0)
    return mean_low, mean_high


# Anchors for GPT latent (if available)
speed_low = speed_high = pitch_low = pitch_high = None
if stats is not None and all_latents is not None:
    speed_low, speed_high = get_anchor_points("cps", all_latents, stats, pct=0.15)
    pitch_low, pitch_high = get_anchor_points("pitch", all_latents, stats, pct=0.15)

# Anchors for speaker embedding (if available)
spk_speed_low = spk_speed_high = spk_pitch_low = spk_pitch_high = None
if stats is not None and all_spk_embs is not None:
    spk_speed_low, spk_speed_high = get_anchor_points("cps", all_spk_embs, stats, pct=0.15)
    spk_pitch_low, spk_pitch_high = get_anchor_points("pitch", all_spk_embs, stats, pct=0.15)


def _require_spk_dirs():
    if spk_speed_direction is None or spk_pitch_direction is None:
        raise RuntimeError(
            "Speaker-embedding directions not found. Re-run direction generation with CREATE_NEW=True and include_speaker=True, "
            "or load a .pt file that contains spk_speed_direction/spk_pitch_direction."
        )


# --- Button handlers ---
def synthesize_linear(_b):
    with ui_out:
        clear_output()

        mode = apply_to.value

        if mode == "gpt":
            modified_latent = gpt_cond_latent + (s_slider.value * speed_direction) + (p_slider.value * pitch_direction)
            modified_spk = speaker_embedding

            print(f"Rendering (Linear, GPT)  S:{s_slider.value}  P:{p_slider.value} ...")
            out = model.inference(
                text="Linear control applied to conditional latents.",
                language="en",
                gpt_cond_latent=modified_latent,
                speaker_embedding=modified_spk,
                temperature=model.config.temperature,
                length_penalty=model.config.length_penalty,
                repetition_penalty=model.config.repetition_penalty,
                top_k=model.config.top_k,
                top_p=model.config.top_p,
            )
            display(Audio(out["wav"], rate=24000, autoplay=True))
            return

        # mode == "spk"
        _require_spk_dirs()

        modified_latent = gpt_cond_latent
        modified_spk = speaker_embedding + (s_slider.value * spk_speed_direction) + (
                    p_slider.value * spk_pitch_direction)

        # Speaker embeddings are typically normalized before inference
        modified_spk = torch.nn.functional.normalize(modified_spk, p=2, dim=1)

        print(f"Rendering (Linear, SPK)  S:{s_slider.value}  P:{p_slider.value} ...")
        out = model.inference(
            text="Linear control applied to speaker embedding.",
            language="en",
            gpt_cond_latent=modified_latent,
            speaker_embedding=modified_spk,
            temperature=model.config.temperature,
            length_penalty=model.config.length_penalty,
            repetition_penalty=model.config.repetition_penalty,
            top_k=model.config.top_k,
            top_p=model.config.top_p,
        )
        display(Audio(out["wav"], rate=24000, autoplay=True))


def synthesize_spherical(_b):
    with ui_out:
        clear_output()

        mode = apply_to.value

        if mode == "gpt":
            if speed_low is None or speed_high is None or pitch_low is None or pitch_high is None:
                raise RuntimeError(
                    "GPT anchor points not available. Generate/load directions with stats/all_latents present.")

            current_latent = gpt_cond_latent

            if s_slider.value != 0:
                target = speed_high if s_slider.value > 0 else speed_low
                strength = abs(s_slider.value) / 5.0
                current_latent = slerp(strength, current_latent, target)

            if p_slider.value != 0:
                target = pitch_high if p_slider.value > 0 else pitch_low
                strength = abs(p_slider.value) / 5.0
                current_latent = slerp(strength, current_latent, target)

            print(f"Rendering (Spherical, GPT)  S:{s_slider.value}  P:{p_slider.value} ...")
            out = model.inference(
                text="Spherical control applied to conditional latents.",
                language="en",
                gpt_cond_latent=current_latent,
                speaker_embedding=speaker_embedding,
                temperature=model.config.temperature,
                length_penalty=model.config.length_penalty,
                repetition_penalty=model.config.repetition_penalty,
                top_k=model.config.top_k,
                top_p=model.config.top_p,
            )
            display(Audio(out["wav"], rate=24000, autoplay=True))
            return

        # mode == "spk"
        if spk_speed_low is None or spk_speed_high is None or spk_pitch_low is None or spk_pitch_high is None:
            raise RuntimeError(
                "SPK anchor points not available. Generate/load directions with stats/all_spk_embs present.")

        current_spk = speaker_embedding

        if s_slider.value != 0:
            target = spk_speed_high if s_slider.value > 0 else spk_speed_low
            strength = abs(s_slider.value) / 5.0
            current_spk = slerp(strength, current_spk, target)

        if p_slider.value != 0:
            target = spk_pitch_high if p_slider.value > 0 else spk_pitch_low
            strength = abs(p_slider.value) / 5.0
            current_spk = slerp(strength, current_spk, target)

        current_spk = torch.nn.functional.normalize(current_spk, p=2, dim=1)

        print(f"Rendering (Spherical, SPK)  S:{s_slider.value}  P:{p_slider.value} ...")
        out = model.inference(
            text="Spherical control applied to speaker embedding.",
            language="en",
            gpt_cond_latent=gpt_cond_latent,
            speaker_embedding=current_spk,
            temperature=model.config.temperature,
            length_penalty=model.config.length_penalty,
            repetition_penalty=model.config.repetition_penalty,
            top_k=model.config.top_k,
            top_p=model.config.top_p,
        )
        display(Audio(out["wav"], rate=24000, autoplay=True))


# Wire up buttons (avoid re-registering multiple times if re-running cell)
linear_btn.on_click(synthesize_linear)
spherical_btn.on_click(synthesize_spherical)

display(
    widgets.VBox(
        [
            apply_to,  # <--- NEW
            s_slider,
            p_slider,
            widgets.HBox([linear_btn, spherical_btn]),
            ui_out,
        ]
    )
)